# Counterfactuals - WhatIf: In-class Exercise 08-2

## Goal

Deepening your understanding of counterfactuals by inspecting another counterfactuals algorithm.

# 1 WhatIf Counterfactuals

Counterfactual explanations are a valuable tool to explain predictions of machine learning models. They tell the user how features need to be changed in order to predict a desired outcome. One of the simplest approaches to generate counterfactuals is to determine, for a given observation `x` (called `x_interest`), the closest data point which has a prediction equal to the desired outcome.

In the following, an implementation of this so-called WhatIf approach for a binary classifier is given:

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator
from typing import Tuple, Optional


def generate_whatif(x_interest: pd.DataFrame, model: BaseEstimator, df: pd.DataFrame) -> Tuple[Optional[pd.Series], Optional[float]]:
    """
    Computes what-if counterfactuals for binary classification models:
    finds the closest data point (using a normalized distance) whose prediction is different.

    Parameters:
      x_interest: A one-row DataFrame representing the data point of interest (including target column).
      model: A binary classifier with a predict method.
      df: The complete dataset (including the target column "Type").

    Returns:
      A tuple (counterfactual, distance) where 'counterfactual' is a Series representing the closest counterfactual row (including target)
      and 'distance' is the computed distance.
    """
    X = df.drop("Type", axis=1)
    
    # Get the prediction for x_interest (dropping the target)
    pred_interest = model.predict(x_interest.drop("Type", axis=1))[0]
    
    preds = model.predict(X)
    
    # Create a boolean mask for those with a different prediction
    mask = preds != pred_interest
    
    # If no counterfactual is found, return None
    if not mask.any():
        return None, None
    
    # Subset the dataframe (all columns) to only those with different predictions
    df_cf = df.loc[mask]
    X_cf = df_cf.drop("Type", axis=1)
    
    # Compute feature ranges from the full dataset X (used for normalization)
    ranges = X.max() - X.min()
    
    xi = x_interest.drop("Type", axis=1).iloc[0]
    
    # Compute the normalized (Gower-like) distance for each row in the counterfactual subset.
    # For numeric features, distance = |x - y| / (max - min), and we average across features.
    distances = X_cf.apply(lambda row: np.mean(np.abs(row - xi) / ranges), axis=1)
    
    # Identify the row with the minimum distance
    min_idx = distances.idxmin()
    return df.loc[min_idx], distances[min_idx]

In this exercise, we consider the wheat seeds dataset (`wheat_seeds.csv`). It consists of seven features distinguishing three types of wheat kernels (0 - Canadian, 1 - Kama, 2 - Rosa). Convert the target `Type` into a binary classification problem to predict whether the wheat kernel belongs to the type *Rosa*. Fit a random forest on the dataset and compute a WhatIf-counterfactual for the first observation.

Your tasks are:

1. Load the dataset and check the class distribution of the target variable `Type`.
2. Convert the classification task into a binary classification task:
   - All samples with `Type = 0` (Canadian) are reassigned to `Type = 1` so that the target indicates whether the kernel is Rosa (`Type = 2`) or not (`Type = 1`).
3. Train a random forest classifier to predict the binary target.
4. Choose the first observation from the dataset as your `x_interest`.
5. Compute a WhatIf-counterfactual for this observation using the provided `generate_whatif` function.

<details><summary>Hint 1:</summary>
To create a binary target, you can transform the `Type` column such that `Type = 0` becomes `1` and the rest stay the same. This will allow you to frame the problem as predicting *Rosa* (Type = 2) vs *not Rosa* (Type = 1).
</details>


In [2]:
#===SOLUTION===

df = pd.read_csv("../data/wheat_seeds.csv")
print(f"Original target value counts:\n{df['Type'].value_counts()}\n")

df["Type"] = df["Type"].apply(lambda x: 1 if x == 0 else x)
print(f"Modified target value counts:\n{df['Type'].value_counts()}\n")

X = df.drop("Type", axis=1)
y = df["Type"]

model = RandomForestClassifier(random_state=42)
model.fit(X, y)

# Choose the first observation as the data point of interest (including its target column)
x_interest = df.iloc[[0]]
print(f"x_interest:\n{x_interest}\n")

# Compute the counterfactual explanation for x_interest
cf, dist = generate_whatif(x_interest, model, df)
if cf is not None:
    print(f"Counterfactual observation (with Gower-like distance {dist:.4f}):\n{cf.to_frame().T}")
else:
    print("No counterfactual found (all observations have the same prediction as x_interest).")


Original target value counts:
Type
2    68
1    66
0    65
Name: count, dtype: int64

Modified target value counts:
Type
1    131
2     68
Name: count, dtype: int64

x_interest:
    Area  Perimeter  Compactness  Kernel.Length  Kernel.Width  \
0  15.26      14.84        0.871          5.763         3.312   

   Asymmetry.Coeff  Kernel.Groove  Type  
0            2.221           5.22     1  

Counterfactual observation (with Gower-like distance 0.0846):
     Area  Perimeter  Compactness  Kernel.Length  Kernel.Width  \
132  15.6      15.11        0.858          5.832         3.286   

     Asymmetry.Coeff  Kernel.Groove  Type  
132            2.725          5.752   2.0  


## 2 Analysis of attributes

Question: Which attributes from the lecture (validity, sparsity, …) does this approach fulfill. Based on this, derive the advantages and disadvantages of the approach.

===SOLUTION===

Counterfactuals generated with WhatIf are valid and proximal, since they reflect the closest training datapoint with the desired/different prediction. The counterfactuals are also plausible since by definition they adhere to the data manifold. The counterfactuals are not sparse and might propose changes to many features - this is a clear disadvantage of this method.

## 3 Evaluation

In order to evaluate the **sparseness** of the counterfactual produced by WhatIf, we can use the following approach: For each feature of the counterfactual instance, assess whether setting its value to the one of `x_interest` still leads to a different prediction than `x_interest`.

Create a function `evaluate_counterfactual()` following these steps:

1. Create an empty list `feature_names`.
2. For each feature:
   - Create a copy of the counterfactual.
   - Replace the feature value in this copy with the value from `x_interest`.
   - Evaluate whether the prediction for this modified copy still differs from the prediction for `x_interest`.
   - If it still differs, add the name of this feature to `feature_names`.
3. Return `feature_names`.

Evaluate the counterfactual derived in the previous exercise using this approach.

<details><summary>Hint 1:</summary>
The function has the following signature and docstring:

```python
def evaluate_counterfactual(counterfactual: pd.Series, x_interest: pd.DataFrame, model) -> list[str]:
    """
    Evaluates whether a counterfactual is minimal.
    For each feature, it replaces the counterfactual's feature value with the corresponding value from x_interest
    and checks if the prediction remains different from x_interest's prediction.
    
    Parameters:
      counterfactual: Counterfactual observation (should not include the target column).
      x_interest: Data point of interest (one-row DataFrame, without the target column).
      model: Binary classifier with a predict method.
      
    Returns:
      List of feature names for which replacing the value with that of x_interest still results in a different prediction.
    """
```
</details>

In [3]:
#===SOLUTION===

def evaluate_counterfactual(counterfactual: pd.Series, x_interest: pd.DataFrame, model) -> list[str]:
    """
    Evaluates whether a counterfactual is minimal.
    For each feature, it replaces the counterfactual's feature value with the corresponding value from x_interest
    and checks if the prediction remains different from x_interest's prediction.
    
    Parameters:
      counterfactual: Counterfactual observation (should not include the target column).
      x_interest: Data point of interest (one-row DataFrame, without the target column).
      model: Binary classifier with a predict method.
      
    Returns:
      List of feature names for which replacing the value with that of x_interest still results in a different prediction.
    """
    pred_interest = model.predict(x_interest)[0]
    feature_names: list[str] = []
    
    # Convert x_interest to a Series (one row) for easier indexing
    x_interest_series = x_interest.iloc[0]
    
    for feature in counterfactual.index:
        # If the feature values are already the same, skip it.
        if counterfactual[feature] == x_interest_series[feature]:
            continue
        
        # Create a copy of the counterfactual and replace the feature with x_interest's value.
        new_cf = counterfactual.copy()
        new_cf[feature] = x_interest_series[feature]
        
        new_pred = model.predict(pd.DataFrame([new_cf]))[0]
        
        # If the prediction remains different, add the feature name.
        if new_pred != pred_interest:
            feature_names.append(feature)
    return feature_names

x_interest_no_target = x_interest.drop(columns=["Type"])
cf_no_target = cf.drop("Type")

# Evaluate the minimality (sparseness) of the counterfactual.
minimal_features = evaluate_counterfactual(cf_no_target, x_interest_no_target, model)
print("Features that still yield a different prediction when set to x_interest's value:")
print(f"{minimal_features}")


Features that still yield a different prediction when set to x_interest's value:
['Area', 'Perimeter', 'Compactness', 'Kernel.Length', 'Kernel.Width', 'Asymmetry.Coeff']


## Summary

We discovered and became familiar with the WhatIf algorithm and its advantages and disadvantages.